# Segmentation and sensor review

23 November 2026

Notes only. Code is in src/preprocess.py and src/eda.py.

## Segment checks

Segments are numbered with (Time.diff() > 1).cumsum(). Numbering starts at zero in each file. Use source filename and segment_id together. The identifier classed as metadata.

| Log | Segment | First time (s) | Last time (s) | Rows |
| --- | ---: | ---: | ---: | ---: |
| synclossexperiment2.csv | 0 | 36.068 | 1402.989 | 20,630 |
| synclossexperiment2.csv | 1 | 2748.031 | 2872.930 | 1,887 |
| night.csv | 0 | 0.000 | 629.369 | 9,510 |
| night.csv | 1 | 1197.157 | 1425.268 | 3,444 |

Checks matched the expected row counts. Segmentation adds one column. Future slopes and rolling features must be calculated within segments.

## Sensor selection and distributions

EDA channels: RPM, MAP, AFR, TPS, IAT, PW, Advance, Dwell, Battery V and Boost PSI. CLT and duty cycle are deferred, not permanently rejected. Final feature choices and exclusions need separate reasons. Original columns remain in the files.

All selected channels contain 150,157 values. Combined histograms use 30 bins. Every row has equal weight, so longer logs contribute more. These are row distributions, not time-weighted distributions.

| Channel | Median | 25% to 75% |
| --- | ---: | --- |
| RPM | 1037 | 982 to 1179 |
| MAP (kPa) | 40 | 38 to 42 |
| AFR | 11.9 | 11.6 to 12.7 |
| TPS (%) | 0 | 0 to 0 |
| PW (ms) | 1.5 | 1.5 to 1.5 |
| Dwell (ms) | 2.0 | 2.0 to 2.0 |

The dataset is strongly shifted towards near idle and low load. High-load and high-RPM examples are sparse. PW and Dwell are concentrated at a few values. AFR values at 20 need operating context checks. Synthetic scenarios can extend experiments, but performance on them does not indicate performance on real faults.

## Extreme-value review

Five largest and five smallest rows per selected channel were located with source filename and Time. Results were saved for later review.

### Abnormal startup Boost PSI

Example: synclossexperiment2.csv contains four MAP = 291 kPa rows at 2752.241, 2752.311, 2752.384 and 2752.454 s. RPM is 241 to 319 and Boost PSI is 27.6. The second start event was at 2751.889 s on a hot engine while the sync loss symptom was present.

The pressure is not realistic for this starting condition. Inspecting with the logger reveals that those are corrupted or interfered sensor readings, and will be eliminated.

## Full-throttle comparison

night.csv around 477 to 481 s was reviewed in MegaLogViewer. TPS rises from roughly 60% to 100%, RPM rises and MAP reaches about 168 to 171 kPa. A gear change is followed by renewed throttle opening and a later throttle lift. The related sensor changes form a logical sequence.

The RPM maximum is 5705 at 479.785 s. A separate point at 544.458 s has MAP = 175 kPa, Boost PSI = 10.9 and RPM = 4314. The 10.9 PSI reading is about 0.75 bar, close to the documented peak pressure of 0.8 bar.

This shows why viewing the sensor values with context is important, when processing and cleaning the data.


## Preliminary decisions

- Exclude startup and shutdown periods from the model dataset. The target is running conditions, and transition readings may be unreliable.
- Review saved extreme values next session before applying anomaly filters.
- Repeated timestamps may be removed or combined in the feature-calculation dataset if needed. The rule is not yet selected. Equal timestamps can contain different sensor readings, so they are not necessarily duplicate measurements.

Next: review anomalies, define transition exclusions, choose duplicate-time handling, then prepare the split and first features.

## Learning and sources

Course chapter 3, Reading and plotting, remains the base for Pandas reading, selection, summaries and plots. Segmentation, pooled-log inspection and review thresholds are project additions.

Exact documentation links are in ../reports/sources_2026-11-23.md.